In [25]:
import re
import html
import numpy as np
import pandas as pd
from datasets import Dataset,ClassLabel
from transformers import (AutoTokenizer,AutoModelForSequenceClassification,TrainingArguments,Trainer,DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score


In [13]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kazanova/sentiment140")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'sentiment140' dataset.
Path to dataset files: /kaggle/input/sentiment140


In [15]:
cols = ["label", "id", "date", "flag", "user", "text"]
df = pd.read_csv(r"/kaggle/input/sentiment140/training.1600000.processed.noemoticon.csv",encoding="latin-1",names=cols, usecols=["label", "text"])

In [20]:
# Remove missing values
df = df.dropna(subset=["text", "label"])
# 0 = Negative
# 4 = Positive
df["label"] = (df["label"] == 4).astype(int)


In [6]:
def clean_transformer(t: str) -> str:
    t = html.unescape(t)
    t = re.sub(r"https?://\S+|www\.\S+",
               "http",t)

    # Mentions
    t = re.sub(r"@\w+","@user",t)

    # Reduce excessive character repetition
    t = re.sub(r"(.)\1{3,}",r"\1\1\1",t)

    # Normalize whitespace
    return re.sub(r"\s+", " ", t).strip()


In [21]:
MODEL = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)

In [22]:
raw = df[["text", "label"]].copy()
raw["text"] = raw["text"].map(clean_transformer)
ds = Dataset.from_pandas(raw,preserve_index=False)

In [26]:
ds = ds.cast_column(
    "label",
    ClassLabel(names=["negative", "positive"])
)
# Stratified split
ds = ds.train_test_split(
    test_size=0.1,
    seed=42,
    stratify_by_column="label"
)

Casting the dataset:   0%|          | 0/1600000 [00:00<?, ? examples/s]

In [27]:
# Optional: limit training size for Colab
MAX_TRAIN = 220_000
if len(ds["train"]) > MAX_TRAIN:
    train_ds = ds["train"].shuffle(seed=42).select(
        range(MAX_TRAIN))
else:
    train_ds = ds["train"]

eval_ds = ds["test"]

In [28]:
def tokenize(batch):
    return tok(
        batch["text"],
        truncation=True,
        max_length=64
    )

train_ds = train_ds.map(
    tokenize,
    batched=True
)

eval_ds = eval_ds.map(
    tokenize,
    batched=True
)


Map:   0%|          | 0/220000 [00:00<?, ? examples/s]

Map:   0%|          | 0/160000 [00:00<?, ? examples/s]

In [29]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=2)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [30]:
# ============================================================
# 6. Metrics
# ============================================================

def metrics(p):

    preds = np.argmax(p.predictions,axis=1)

    return {
        "accuracy": accuracy_score(
            p.label_ids,
            preds
        ),
        "f1": f1_score(
            p.label_ids,
            preds,
            average="binary"
        )
    }

In [32]:
args = TrainingArguments(
    output_dir="distilbert_sentiment",

    num_train_epochs=2,

    learning_rate=2e-5,

    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,

    weight_decay=0.01,

    fp16=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,

    seed=42,

    report_to="none"
)



In [33]:
trainer = Trainer(
    model=model,
    args=args,

    train_dataset=train_ds,
    eval_dataset=eval_ds,

    data_collator=DataCollatorWithPadding(tok),

    compute_metrics=metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.354757,0.350162,0.847500,0.843499
2,0.306468,0.344099,0.852200,0.850369


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6876, training_loss=0.34275355846677985, metrics={'train_runtime': 614.9975, 'train_samples_per_second': 715.45, 'train_steps_per_second': 11.181, 'total_flos': 4989839927061120.0, 'train_loss': 0.34275355846677985, 'epoch': 2.0})

In [34]:
trainer.save_model("distilbert_sentiment_final")
tok.save_pretrained("distilbert_sentiment_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('distilbert_sentiment_final/tokenizer_config.json',
 'distilbert_sentiment_final/tokenizer.json')

In [36]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_PATH = "/content/distilbert_sentiment_final"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

model.eval()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [37]:
text = "I really love this product!"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=64
)

with torch.no_grad():
    outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

print(prediction)

1


In [38]:
label = "positive" if prediction == 1 else "negative"

print("Prediction:", label)

Prediction: positive
